# Data description

In [5]:

import os
import sys
import importlib
from importlib import reload 
from dataclasses import dataclass, field, fields
from itertools import compress
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from cycler import cycler

import scipy
import scipy.io as sio
from scipy import signal
from scipy.signal import spectrogram, hann, butter, filtfilt, freqz
from scipy import stats
from scipy.stats import norm
from scipy.stats import ttest_ind

import seaborn as sns
import pingouin as pg
import itertools
from itertools import combinations
from statannotations.Annotator import Annotator
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import plotly.express as px

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report

# import openpyxl
# from openpyxl import Workbook, load_workbook
# import xlrd
import pickle
import json
import csv

#mne
import mne_bids
import mne
from mne.time_frequency import tfr_morlet 
from mne.stats import permutation_cluster_test


# TODO: add README file: 
# pip install pingouin
# pip install statannotations
# pip install fooof

In [36]:
jennifer_user_path = os.getcwd()
while jennifer_user_path[-14:] != 'jenniferbehnke':
    jennifer_user_path = os.path.dirname(jennifer_user_path)

# directory to this Repository
project_path = os.path.join(jennifer_user_path, 'code', 'BetaSenSightLongterm', 'BetaSenSightLongterm')
sys.path.append(project_path)

os.chdir(project_path)

import src.bssu.utils.find_folders as find_folders
importlib.reload(find_folders)

# import PyPerceive

project_path = find_folders.chdir_repository("Py_Perceive")

from PerceiveImport.classes import (
    main_class, modality_class, metadata_class,
    session_class, condition_class, task_class,
    contact_class, run_class
)

import PerceiveImport.methods.load_rawfile as load_rawfile
import PerceiveImport.methods.find_folders as PyPerceive_find_folders
import PerceiveImport.methods.metadata_helpers as metaHelpers

# import meet

project_path = find_folders.chdir_repository("meet")

import meet as meet

# import all functions from BetaSenSightLongterm
project_path = find_folders.chdir_repository("BetaSenSightLongterm")

# tfr, processing
import src.bssu.tfr.BSSuPsd as BSSuPsd
import src.bssu.tfr.FastFourierPSD as FFpsd

# bipolar Channel Analysis
import src.bssu.bipolar.power_spectra_plots as power_spectra_plots
import src.bssu.bipolar.PeakFrequencies_PSD as PeakFrequency_psd
import src.bssu.bipolar.BIP_channelGroups as BIP_channelGroups
import src.bssu.bipolar.BIP_perChannelAnalysis as BIP_perChannel


# monopolar Referencing
import src.bssu.monopolar.MonoRef_JLB as MonoRefJLB
import src.bssu.monopolar.GroupMonopolarPSD as groupMonopol
import src.bssu.monopolar.monoRef_weightPsdAverageByCoordinateDistance as MonoRefWeightedCoordinateDistance
import src.bssu.monopolar.externalized_lfp as externalized
import src.bssu.monopolar.monopol_method_comparison as monopol_comparison
import src.bssu.monopolar.bssu_contacts_maximal_beta as bssu_contacts
import src.bssu.monopolar.monoRef_Strelow as detec_strelow

# Ranking Order
import src.bssu.ranking.HighestRankedChannelPSD as highestRank
import src.bssu.ranking.monopolPSDaverage_withinSubject as PSDaverageMonopol
import src.bssu.ranking.BIPchannelGroups_ranks as BIP_ranks
import src.bssu.ranking.Permutation_rankings as Permute_ranks
import src.bssu.ranking.beta_rank_longitudinal_changes as beta_rank_change


# Clinical stimulation parameters
import src.bssu.stimulation.activeStimulationContacts as activeStimContacts

# utility functions
import src.bssu.utils.loadResults as loadResults
import src.bssu.utils.find_folders as find_folders
import src.bssu.utils.writeGroupDataframes as writeGroupDF
import src.bssu.utils.load_data_files as load_data
import src.bssu.utils.monopol_comparison_helpers as mono_comp_helpers
import src.bssu.utils.sub_session_dict as sub_session_dict
# import Classes
from src.bssu.classes import (metadataAnalysis_class, mainAnalysis_class, sessionAnalysis_class, 
                              channelAnalysis_class, featureAnalysis_class, frequencyBand_class)

# import mni coordinates
import src.bssu.mni.load_rotated_coordinates as load_mni


importlib.reload(BSSuPsd)
importlib.reload(MonoRefJLB)
importlib.reload(loadResults)
importlib.reload(highestRank)
importlib.reload(groupMonopol)
importlib.reload(PSDaverageMonopol)
importlib.reload(FFpsd)
importlib.reload(find_folders)
importlib.reload(metadataAnalysis_class)
importlib.reload(mainAnalysis_class)
importlib.reload(sessionAnalysis_class)
importlib.reload(channelAnalysis_class)
importlib.reload(featureAnalysis_class)
importlib.reload(frequencyBand_class)
importlib.reload(PeakFrequency_psd)
importlib.reload(power_spectra_plots)
importlib.reload(BIP_channelGroups)
importlib.reload(BIP_ranks)
importlib.reload(activeStimContacts)
importlib.reload(Permute_ranks)
importlib.reload(BIP_perChannel)
importlib.reload(load_mni)
importlib.reload(writeGroupDF)
importlib.reload(MonoRefWeightedCoordinateDistance)
importlib.reload(load_data)
importlib.reload(externalized)
importlib.reload(monopol_comparison)
importlib.reload(bssu_contacts)
importlib.reload(mono_comp_helpers)
importlib.reload(detec_strelow)
importlib.reload(beta_rank_change)
importlib.reload(sub_session_dict)

Excel file loaded:  patient_metadata.xlsx 
loaded from:  /Users/jenniferbehnke/Dropbox/work/ResearchProjects/Monopolar_power_estimation/data


<module 'src.bssu.utils.sub_session_dict' from '/Users/jenniferbehnke/code/BetaSenSightLongterm/BetaSenSightLongterm/src/bssu/utils/sub_session_dict.py'>

## Steps to write data

## Load finalized FOOOF data

### FOOOF data per group

In [15]:
# FOOOF data of all channels (15 channels per STN)

beta_rank_DF = loadResults.load_fooof_beta_ranks(
        fooof_spectrum="periodic_spectrum",
        fooof_version="v2",
        all_or_one_chan="beta_all",
        all_or_one_longterm_ses="one_longterm_session",
    )

beta_rank_DF.head()

,subject_hemisphere,session,bipolar_channel,fooof_error,fooof_r_sq,fooof_exponent,fooof_offset,fooof_power_spectrum,periodic_plus_aperiodic_power_log,fooof_periodic_flat,fooof_number_peaks,alpha_peak_CF_power_bandWidth,low_beta_peak_CF_power_bandWidth,high_beta_peak_CF_power_bandWidth,beta_peak_CF_power_bandWidth,gamma_peak_CF_power_bandWidth,beta_average
0,017_Right,fu3m,03,0.066416,0.98557,1.688615,1.332174,"[0.00767014426514212, 0.03351470878412366, 0.1...","[0.8243496620189412, 0.5308087522174064, 0.339...","[0.0004994452423981573, 0.004308872706144536, ...",4,"[nan, nan, nan]","[nan, nan, nan]","[28.987529527052935, 0.5952799100446403, 7.519...","[28.987529527052935, 0.5952799100446403, 7.519...","[nan, nan, nan]",0.073489
1,017_Right,fu3m,13,0.088264,0.96624,1.457102,1.030205,"[0.008159401337378469, 0.006371132823893344, 0...","[0.5924797149155441, 0.33626772579620084, 0.15...","[0.0009066008320197118, 0.0012775321970460254,...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.225692476850114, 0.9213419411183563, 3.000...","[29.225692476850114, 0.9213419411183563, 3.000...","[nan, nan, nan]",0.150319
2,017_Right,fu3m,02,0.093521,0.971951,1.685157,1.420904,"[7.787614109133756e-09, 1.882592837176844e-08,...","[0.9136209432416069, 0.6168795782504854, 0.406...","[4.126367042459155e-10, 1.975427741277798e-09,...",3,"[nan, nan, nan]","[nan, nan, nan]","[29.073003972962415, 0.9881124104459198, 8.238...","[29.073003972962415, 0.9881124104459198, 8.238...","[nan, nan, nan]",0.254790
3,017_Right,fu3m,12,0.07857,0.972904,1.343522,0.757904,"[0.0003292927862230677, 0.0003079802974601531,...","[0.35352735458664525, 0.11698373243642358, -0....","[6.336812502902425e-05, 0.00010218204803303865...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.43438184459086, 1.099716477148555, 4.71406...","[29.43438184459086, 1.099716477148555, 4.71406...","[nan, nan, nan]",0.204537
4,017_Right,fu3m,01,0.071117,0.979469,1.587926,1.157176,"[0.0260612717972748, 0.08068602780119916, 0.20...","[0.6815260508032184, 0.4132884933120965, 0.253...","[0.0023628423792285266, 0.01374512167041695, 0...",4,"[nan, nan, nan]","[nan, nan, nan]","[33.50000000000001, 0.34359567811682745, 3.000...","[33.50000000000001, 0.34359567811682745, 3.000...","[64.15854737876835, 0.19383025351969008, 10.73...",0.030401


In [16]:
beta_rank_DF.columns

Index(['subject_hemisphere', 'session', 'bipolar_channel', 'fooof_error',
       'fooof_r_sq', 'fooof_exponent', 'fooof_offset', 'fooof_power_spectrum',
       'periodic_plus_aperiodic_power_log', 'fooof_periodic_flat',
       'fooof_number_peaks', 'alpha_peak_CF_power_bandWidth',
       'low_beta_peak_CF_power_bandWidth', 'high_beta_peak_CF_power_bandWidth',
       'beta_peak_CF_power_bandWidth', 'gamma_peak_CF_power_bandWidth',
       'beta_average'],
      dtype='object')

Data description of all included FOOOF channels

In [51]:
def select_fooof_data(cohort:str):
    """
    Select only data from a specific cohort of subjects

    Input:
    cohort: str
        cohort of subjects, e.g. "all_included", "group_1", "group_2", "group_3"
    """

    fooof_data = loadResults.load_fooof_beta_ranks(
            fooof_spectrum="periodic_spectrum",
            fooof_version="v2",
            all_or_one_chan="beta_all",
            all_or_one_longterm_ses="one_longterm_session",
        )

    # select only data from included recordings
    included_sub_sessions = sub_session_dict.get_subs_sessions(cohort) # other options: "group_1", "group_2", "group_3"
    incl_subjects = list(included_sub_sessions["incl_subjects"]) # list of subjects
    sub_sessions = included_sub_sessions["incl_sessions"] # dict with keys subjects, sessions values

    # sub_hem list, e.g. "017_Right"
    sub_hem_list = [f"{sub}_{side}" for sub in incl_subjects for side in ("Left", "Right")]

    # only keep rows of dataframe if value is in list subject
    return fooof_data[fooof_data["subject_hemisphere"].isin(sub_hem_list)]
    

In [53]:
selected_fooof_data = select_fooof_data("all_included")
selected_fooof_data

,subject_hemisphere,session,bipolar_channel,fooof_error,fooof_r_sq,fooof_exponent,fooof_offset,fooof_power_spectrum,periodic_plus_aperiodic_power_log,fooof_periodic_flat,fooof_number_peaks,alpha_peak_CF_power_bandWidth,low_beta_peak_CF_power_bandWidth,high_beta_peak_CF_power_bandWidth,beta_peak_CF_power_bandWidth,gamma_peak_CF_power_bandWidth,beta_average
0,017_Right,fu3m,03,0.066416,0.98557,1.688615,1.332174,"[0.00767014426514212, 0.03351470878412366, 0.1...","[0.8243496620189412, 0.5308087522174064, 0.339...","[0.0004994452423981573, 0.004308872706144536, ...",4,"[nan, nan, nan]","[nan, nan, nan]","[28.987529527052935, 0.5952799100446403, 7.519...","[28.987529527052935, 0.5952799100446403, 7.519...","[nan, nan, nan]",0.073489
1,017_Right,fu3m,13,0.088264,0.96624,1.457102,1.030205,"[0.008159401337378469, 0.006371132823893344, 0...","[0.5924797149155441, 0.33626772579620084, 0.15...","[0.0009066008320197118, 0.0012775321970460254,...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.225692476850114, 0.9213419411183563, 3.000...","[29.225692476850114, 0.9213419411183563, 3.000...","[nan, nan, nan]",0.150319
2,017_Right,fu3m,02,0.093521,0.971951,1.685157,1.420904,"[7.787614109133756e-09, 1.882592837176844e-08,...","[0.9136209432416069, 0.6168795782504854, 0.406...","[4.126367042459155e-10, 1.975427741277798e-09,...",3,"[nan, nan, nan]","[nan, nan, nan]","[29.073003972962415, 0.9881124104459198, 8.238...","[29.073003972962415, 0.9881124104459198, 8.238...","[nan, nan, nan]",0.254790
3,017_Right,fu3m,12,0.07857,0.972904,1.343522,0.757904,"[0.0003292927862230677, 0.0003079802974601531,...","[0.35352735458664525, 0.11698373243642358, -0....","[6.336812502902425e-05, 0.00010218204803303865...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.43438184459086, 1.099716477148555, 4.71406...","[29.43438184459086, 1.099716477148555, 4.71406...","[nan, nan, nan]",0.204537
4,017_Right,fu3m,01,0.071117,0.979469,1.587926,1.157176,"[0.0260612717972748, 0.08068602780119916, 0.20...","[0.6815260508032184, 0.4132884933120965, 0.253...","[0.0023628423792285266, 0.01374512167041695, 0...",4,"[nan, nan, nan]","[nan, nan, nan]","[33.50000000000001, 0.34359567811682745, 3.000...","[33.50000000000001, 0.34359567811682745, 3.000...","[64.15854737876835, 0.19383025351969008, 10.73...",0.030401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3505,084_Left,fu12m,2B2C,0.074114,0.97023,0.964667,0.46308,"[0.004093678962242331, 0.0063080130345265495, ...","[0.1738797507599967, 0.005530796411454526, -0....","[0.0011929292013696255, 0.0027133264839467723,...",4,"[nan, nan, nan]","[17.331240781811804, 0.8031401314431682, 8.496...","[22.863866719624518, 0.6928315693108564, 3.000...","[17.331240781811804, 0.8031401314431682, 8.496...","[nan, nan, nan]",0.396047
3506,084_Left,fu12m,2A2C,0.073069,0.967592,1.028199,0.911906,"[3.1242958620225636e-07, 6.738843918441262e-06...","[0.602387513661082, 0.42133166458963356, 0.292...","[3.3896040284148175e-08, 1.109273400161362e-06...",4,"[10.429125808490593, 0.23223723564464602, 3.00...","[17.19128095622085, 1.0531034396978112, 4.7915...","[22.041871042441635, 0.7548574255563847, 3.000...","[17.19128095622085, 1.0531034396978112, 4.7915...","[nan, nan, nan]",1.201494
3507,084_Left,fu12m,1A2A,0.069376,0.983246,1.499076,1.130707,"[0.3901233117377263, 0.28653634934393724, 0.24...","[0.7135120128380136, 0.46082048997325104, 0.28...","[0.03407209804212193, 0.04535483854955621, 0.0...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.947727153146662, 0.49035522341896054, 3.75...","[20.79440718481877, 0.5389240464429107, 15.994...","[nan, nan, nan]",0.223957
3508,084_Left,fu12m,1B2B,0.075526,0.978354,1.226906,0.258918,"[0.0024234526213646745, 0.0027680048514447386,...","[-0.1090624406261136, -0.32392313175726706, -0...","[0.0013550603558905392, 0.0025418187532027965,...",4,"[nan, nan, nan]","[16.11044561637067, 1.0985351080232724, 3.0981...","[22.305150266421336, 0.944265515944136, 11.222...","[16.11044561637067, 1.098535108023

In [57]:
### data description
selected_fooof_data = select_fooof_data("all_included")

# list of subjects
subjects = list(selected_fooof_data.subject_hemisphere.str.split("_").str[0].unique())
sub_hem_list = list(selected_fooof_data.subject_hemisphere.unique())

print(
    f"Number of channels fitted: {len(selected_fooof_data.subject_hemisphere.values)}"
)
print(f"List of STNs: {list(selected_fooof_data.subject_hemisphere.unique())}")
print(
    f"Number of STNs: {len(selected_fooof_data.subject_hemisphere.unique())}",
    "\nNumber of patients: "
    + (str(len(selected_fooof_data.subject_hemisphere.unique()) / 2)),
)

# mean, std, sem of fooof error and r squared
mean_fooof_error = np.mean(selected_fooof_data["fooof_error"])
std_fooof_error = np.std(selected_fooof_data["fooof_error"])
sem_fooof_error = stats.sem(selected_fooof_data["fooof_error"])

mean_fooof_r_sq = np.mean(selected_fooof_data["fooof_r_sq"])
std_fooof_r_sq = np.std(selected_fooof_data["fooof_r_sq"])
sem_fooof_r_sq = stats.sem(selected_fooof_data["fooof_r_sq"])

print(f"FOOOF error mean: {mean_fooof_error},",
      f"\nFOOOF error standard deviation: {std_fooof_error},",
      f"\nFOOOF error sem: {sem_fooof_error},",

      f"\n\nFOOOF r squared mean: {mean_fooof_r_sq}",
      f"\nFOOOF r squared standard deviation: {std_fooof_r_sq},",
      f"\nFOOOF r squared sem: {sem_fooof_r_sq}")

Number of channels fitted: 2220
List of STNs: ['017_Right', '017_Left', '019_Right', '019_Left', '021_Right', '021_Left', '024_Right', '024_Left', '025_Right', '025_Left', '026_Right', '026_Left', '029_Right', '029_Left', '030_Right', '030_Left', '033_Right', '033_Left', '040_Right', '040_Left', '041_Right', '041_Left', '050_Right', '050_Left', '059_Right', '059_Left', '060_Right', '060_Left', '061_Right', '061_Left', '062_Right', '062_Left', '063_Right', '063_Left', '066_Right', '066_Left', '069_Right', '069_Left', '072_Right', '072_Left', '075_Right', '075_Left', '081_Right', '081_Left', '084_Right', '084_Left']
Number of STNs: 46 
Number of patients: 23.0
FOOOF error mean: 0.07597631792190303, 
FOOOF error standard deviation: 0.012899174216733316, 
FOOOF error sem: 0.0002738314065990877, 

FOOOF r squared mean: 0.9680899429728431 
FOOOF r squared standard deviation: 0.02318105535415313, 
FOOOF r squared sem: 0.0004921013459795417


### BIPOLAR

In [13]:
# FOOOF data only of one longterm session
# 12 LFP channels per STN

beta_rank_DF = loadResults.load_fooof_beta_ranks(
        fooof_spectrum="periodic_spectrum",
        fooof_version="v2",
        all_or_one_chan="beta_ranks_all",
        all_or_one_longterm_ses="one_longterm_session",
    )

beta_rank_DF.head()

,index,subject_hemisphere,session,bipolar_channel,fooof_error,fooof_r_sq,fooof_exponent,fooof_offset,fooof_power_spectrum,periodic_plus_aperiodic_power_log,fooof_periodic_flat,fooof_number_peaks,alpha_peak_CF_power_bandWidth,low_beta_peak_CF_power_bandWidth,high_beta_peak_CF_power_bandWidth,beta_peak_CF_power_bandWidth,gamma_peak_CF_power_bandWidth,beta_average,beta_rank
0,3,017_Right,fu3m,12,0.07857,0.972904,1.343522,0.757904,"[0.0003292927862230677, 0.0003079802974601531,...","[0.35352735458664525, 0.11698373243642358, -0....","[6.336812502902425e-05, 0.00010218204803303865...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.43438184459086, 1.099716477148555, 4.71406...","[29.43438184459086, 1.099716477148555, 4.71406...","[nan, nan, nan]",0.204537,1.0
1,4,017_Right,fu3m,01,0.071117,0.979469,1.587926,1.157176,"[0.0260612717972748, 0.08068602780119916, 0.20...","[0.6815260508032184, 0.4132884933120965, 0.253...","[0.0023628423792285266, 0.01374512167041695, 0...",4,"[nan, nan, nan]","[nan, nan, nan]","[33.50000000000001, 0.34359567811682745, 3.000...","[33.50000000000001, 0.34359567811682745, 3.000...","[64.15854737876835, 0.19383025351969008, 10.73...",0.030401,3.0
2,5,017_Right,fu3m,23,0.075302,0.975031,1.291468,0.634505,"[3.658637837133938e-10, 8.781297688997824e-09,...","[0.24573455686045234, 0.018318407410551848, -0...","[9.023380458332988e-11, 3.6561550966272852e-09...",3,"[nan, nan, nan]","[13.602286189596768, 0.49020364390555904, 3.46...","[30.219295965656492, 1.0778976327745415, 6.289...","[30.219295965656492, 1.0778976327745415, 6.289...","[nan, nan, nan]",0.193264,2.0
0,12,017_Right,fu3m,1A2A,0.086413,0.969713,1.477671,1.203942,"[1.051259040174557e-05, 9.481947880196628e-06,...","[0.7591189582797522, 0.4989144774934115, 0.314...","[7.950153134128949e-07, 1.3054734695782597e-06...",3,"[nan, nan, nan]","[nan, nan, nan]","[30.823647968559502, 1.0136923869600145, 7.425...","[30.823647968559502, 1.0136923869600145, 7.425...","[nan, nan, nan]",0.277904,1.0
1,13,017_Right,fu3m,1B2B,0.074097,0.965587,1.172131,0.590142,"[0.008637710295463563, 0.008756105291477745, 0...","[0.23946149626184762, 0.034419922498690476, -0...","[0.0021667278346138965, 0.0035272626669674077,...",4,"[nan, nan, nan]","[17.35611052694957, 0.1139847371212338, 11.042...","[31.41707924195963, 0.9693953353238327, 4.1148...","[31.41707924195963, 0.9693953353238327, 4.1148...","[nan, nan, nan]",0.158807,2.0


In [12]:
# FOOOF data only of one longterm session
# only highest beta channel (rank 1) per LFP group

beta_rank_DF = loadResults.load_fooof_beta_ranks(
        fooof_spectrum="periodic_spectrum",
        fooof_version="v2",
        all_or_one_chan="highest_beta",
        all_or_one_longterm_ses="one_longterm_session",
    )

beta_rank_DF.head()

,index,subject_hemisphere,session,bipolar_channel,fooof_error,fooof_r_sq,fooof_exponent,fooof_offset,fooof_power_spectrum,periodic_plus_aperiodic_power_log,fooof_periodic_flat,fooof_number_peaks,alpha_peak_CF_power_bandWidth,low_beta_peak_CF_power_bandWidth,high_beta_peak_CF_power_bandWidth,beta_peak_CF_power_bandWidth,gamma_peak_CF_power_bandWidth,beta_average,beta_rank
0,3,017_Right,fu3m,12,0.07857,0.972904,1.343522,0.757904,"[0.0003292927862230677, 0.0003079802974601531,...","[0.35352735458664525, 0.11698373243642358, -0....","[6.336812502902425e-05, 0.00010218204803303865...",4,"[nan, nan, nan]","[nan, nan, nan]","[29.43438184459086, 1.099716477148555, 4.71406...","[29.43438184459086, 1.099716477148555, 4.71406...","[nan, nan, nan]",0.204537,1.0
0,12,017_Right,fu3m,1A2A,0.086413,0.969713,1.477671,1.203942,"[1.051259040174557e-05, 9.481947880196628e-06,...","[0.7591189582797522, 0.4989144774934115, 0.314...","[7.950153134128949e-07, 1.3054734695782597e-06...",3,"[nan, nan, nan]","[nan, nan, nan]","[30.823647968559502, 1.0136923869600145, 7.425...","[30.823647968559502, 1.0136923869600145, 7.425...","[nan, nan, nan]",0.277904,1.0
5,11,017_Right,fu3m,2A2C,0.070059,0.975923,1.279358,0.622745,"[1.3979065716096528e-06, 2.5015002960149246e-0...","[0.2376206914896252, 0.01233768192489373, -0.1...","[3.512707128119114e-07, 1.0559606045368374e-06...",4,"[nan, nan, nan]","[13.59303036339205, 0.12463061825765231, 3.000...","[28.344306091350532, 0.9307513411542273, 9.686...","[28.344306091350532, 0.9307513411542273, 9.686...","[nan, nan, nan]",0.172262,1.0
2,20,017_Right,fu12m,23,0.064378,0.989523,1.700796,1.440666,"[2.6991400101650243e-05, 5.4799935694127555e-0...","[0.9286766131841131, 0.6291855485581883, 0.416...","[1.3814456454305008e-06, 5.589611276521289e-06...",4,"[nan, nan, nan]","[13.805828105459103, 0.7460678514380996, 3.064...","[28.513423058870664, 0.9111804619360752, 5.374...","[28.513423058870664, 0.9111804619360752, 5.374...","[nan, nan, nan]",0.516692,1.0
0,27,017_Right,fu12m,1A2A,0.054754,0.98773,1.600978,1.328914,"[0.0011917227674977937, 0.0012032524253271504,...","[0.8470456604000512, 0.5651960382547497, 0.365...","[7.36123518097955e-05, 0.00014223816337737217,...",4,"[nan, nan, nan]","[nan, nan, nan]","[27.185455000237216, 0.41889408694839625, 12.0...","[27.185455000237216, 0.41889408694839625, 12.0...","[63.12950075977409, 0.2016035648850072, 3.0000...",0.111572,1.0


### MONOPOLAR

In [4]:
loaded_fooof_monopolar = loadResults.load_pickle_group_result(
        filename=f"fooof_monoRef_all_contacts_weight_beta_psd_by_inverse_sq_distance_v2",
        fooof_version="v2",
    )
loaded_fooof_monopolar.head()

,index,coord_z,coord_xy,session,subject_hemisphere,estimated_monopolar_beta_psd,contact,beta_relative_to_max,beta_cluster,rank_8,beta_psd_rel_to_rank1,beta_psd_rel_range_0_to_1
0,422,2.0+0.0j,0.650000+0.000000j,fu3m,017_Left,0.563519,1A,0.666290,2,5.0,0.666290,0.614419
1,423,2.0+0.0j,-0.325000+0.562917j,fu3m,017_Left,0.553585,1B,0.654544,2,6.0,0.654544,0.600847
2,424,2.0+0.0j,-0.325000-0.562917j,fu3m,017_Left,0.599291,1C,0.708586,1,4.0,0.708586,0.663289
3,425,4.0+0.0j,0.650000+0.000000j,fu3m,017_Left,0.845756,2A,1.000000,1,1.0,1.000000,1.000000
4,426,4.0+0.0j,-0.325000+0.562917j,fu3m,017_Left,0.648357,2B,0.766600,1,3.0,0.766600,0.730321
